# 53 — Design Principles of Region-Constrained Search

**Thesis**

Physics specifies admissibility.  
Admissibility specifies topology.  
Topology specifies robustness.  
Robustness specifies operation.  
Operation specifies computation.

Notebook 49 established the method. Notebook 53 states the engineering principles behind it.

## Runtime requirements

This notebook uses `numpy`, `pandas`, `matplotlib`, `scipy`, and `IPython`.

In Colab these are usually available. Locally:

```bash
pip install numpy pandas matplotlib scipy ipython
```

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, Ellipse, Polygon
from scipy import ndimage

OUT = Path("outputs/notebook_53_design_principles")
FIG = OUT / "figures"
DATA = OUT / "data"
FIG.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 11,
    "axes.titlesize": 19,
    "axes.labelsize": 12,
})

def savefig(fig, name):
    path = FIG / name
    fig.savefig(path, bbox_inches="tight")
    return path

def clean_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def box(ax, x, y, w, h, title, subtitle="", lw=2.0, rounded=False, title_size=11, subtitle_size=8):
    patch_cls = FancyBboxPatch if rounded else Rectangle
    if rounded:
        patch = patch_cls((x,y), w,h, boxstyle="round,pad=0.02,rounding_size=0.025",
                          fill=False, linewidth=lw, edgecolor="black")
    else:
        patch = patch_cls((x,y), w,h, fill=False, linewidth=lw, edgecolor="black")
    ax.add_patch(patch)
    ax.text(x+w/2, y+h*0.62, title, ha="center", va="center", fontsize=title_size, fontweight="bold")
    if subtitle:
        ax.text(x+w/2, y+h*0.27, subtitle, ha="center", va="center", fontsize=subtitle_size)
    return patch

def arrow(ax, start, end, lw=2.0, color="black"):
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->", linewidth=lw, color=color, shrinkA=0, shrinkB=0))

def vertical_chain(ax, xs, y0, dy, labels, title=None, box_w=0.34, box_h=0.075, title_size=11):
    if title:
        ax.text(xs, y0 + 0.12, title, ha="center", fontsize=14, fontweight="bold")
    for i, (t, s) in enumerate(labels):
        y = y0 - i*dy
        box(ax, xs-box_w/2, y, box_w, box_h, t, s, title_size=title_size)
        if i < len(labels)-1:
            arrow(ax, (xs, y), (xs, y-dy+box_h), lw=1.8)

## 00 Context

A region-constrained design begins by asking what is admissible before selecting a point.

In [ ]:
# Synthetic geometry used only for conceptual figures.
N = 260
drive = np.linspace(0, 1, N)
loss = np.linspace(0, 1, N)
X, Y = np.meshgrid(drive, loss)
threshold = 0.36

def sigmoid(z, s=0.05):
    return 1 / (1 + np.exp(-z / s))

def support_field(center=0.62, width=0.30, loss_scale=0.22, shoulder_amp=0.38, shoulder_x=0.80, shoulder_y=0.10):
    plateau = sigmoid(X - (center - width), 0.045) * sigmoid((center + width) - X, 0.045)
    low_loss = np.exp(-(Y / loss_scale) ** 1.55)
    shoulder = shoulder_amp * np.exp(-((X - shoulder_x)**2 / 0.025 + (Y - shoulder_y)**2 / 0.020))
    return np.clip(plateau * low_loss + shoulder, 0, 1)

def region_metrics(Z, threshold=threshold):
    mask = Z >= threshold
    labels, ncomp = ndimage.label(mask)
    if ncomp == 0:
        return dict(mask=mask, largest=np.zeros_like(mask), dist=np.zeros_like(Z), ncomp=0, area=0, margin=0, x=np.nan, y=np.nan)
    sizes = ndimage.sum(mask, labels, index=np.arange(1, ncomp+1))
    largest_id = int(np.argmax(sizes)+1)
    largest = labels == largest_id
    dist_pix = ndimage.distance_transform_edt(largest)
    dist = dist_pix / dist_pix.max() if dist_pix.max() else dist_pix
    idx = np.unravel_index(np.argmax(dist_pix), dist_pix.shape)
    return dict(mask=mask, largest=largest, dist=dist, ncomp=int(ncomp),
                area=float(largest.mean()), margin=float(dist_pix.max()/N),
                x=float(drive[idx[1]]), y=float(loss[idx[0]]))

Z = support_field()
M = region_metrics(Z)
summary_rows = [
    {"principle": 1, "statement": "Physics specifies admissibility.", "object": "Ω"},
    {"principle": 2, "statement": "Admissibility specifies topology.", "object": "Ωc"},
    {"principle": 3, "statement": "Topology specifies robustness.", "object": "d(∂Ωc)"},
    {"principle": 4, "statement": "Robustness specifies operation.", "object": "x*"},
    {"principle": 5, "statement": "Operation specifies computation.", "object": "fault-tolerant execution"},
]
principles = pd.DataFrame(summary_rows)
principles.to_csv(DATA / "53_design_principles.csv", index=False)
principles

In [ ]:
fig, ax = plt.subplots(figsize=(13,4.8))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Region-Constrained Search: Design Principle")
steps=[
    ("Physics","what can exist?"),
    ("Admissibility","what survives?"),
    ("Topology","what remains connected?"),
    ("Robustness","what has margin?"),
    ("Operation","what can execute?"),
    ("Computation","what remains fault-tolerant?"),
]
xs=np.linspace(0.06,0.82,len(steps)); w,h=0.12,0.22
for i,(t,s) in enumerate(steps):
    box(ax,xs[i],0.47,w,h,t,s,lw=2.3,rounded=True,title_size=10,subtitle_size=8)
    if i < len(steps)-1:
        arrow(ax,(xs[i]+w+0.012,0.58),(xs[i+1]-0.012,0.58),lw=2.0)
ax.text(0.5,0.22,"The specification is the region; the operating point is one admitted realization inside it.",ha="center",fontsize=13)
savefig(fig,"53_01_region_constrained_design_principle.png")
plt.show()

## 07 Leading vs Trailing Specifications

Leading specifications determine what counts as valid. Trailing optimizations summarize a score after constraints have already selected the search space.

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Leading Specifications vs Trailing Optimization")
left=[("Physics","resources"),("Admissibility","Ω"),("Topology","Ωc"),("Robustness","d(∂Ωc)"),("Operation","x*")]
right=[("Objective","summary score"),("Gradient","local direction"),("Local optimum","point"),("Metric report","after the fact")]
vertical_chain(ax,0.30,0.74,0.13,left,title="Leading specification",box_w=0.36,box_h=0.075)
vertical_chain(ax,0.73,0.69,0.13,right,title="Trailing optimization",box_w=0.32,box_h=0.075)
ax.text(0.50,0.08,"Leading specifications constrain what counts as a valid search space. Trailing metrics summarize what happened.",ha="center",fontsize=12)
savefig(fig,"53_02_leading_vs_trailing_specifications.png")
plt.show()

## 13 Constraint Hierarchy

Topology is downstream from physics and upstream from optimization.

In [ ]:
fig, ax = plt.subplots(figsize=(11,6))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Constraint Hierarchy")
box(ax,0.38,0.78,0.24,0.09,"Physics","allowed resources",rounded=True,lw=2.4)
children=[("hardware",0.12),("fabrication",0.32),("energy",0.52),("control",0.72)]
for label,x in children:
    box(ax,x,0.60,0.16,0.08,label,"constraint",rounded=True,lw=1.6,title_size=9)
    arrow(ax,(0.50,0.78),(x+0.08,0.68),lw=1.4)
chain=[("Admissibility","Ω"),("Topology","Ωc"),("Robustness","distance to boundary"),("Operation","x*")]
ys=[0.43,0.31,0.19,0.07]
for (t,s),y in zip(chain,ys):
    box(ax,0.35,y,0.30,0.075,t,s,rounded=True,lw=2.0,title_size=10,subtitle_size=8)
for y1,y2 in zip(ys[:-1],ys[1:]):
    arrow(ax,(0.50,y1),(0.50,y2+0.075),lw=1.7)
arrow(ax,(0.50,0.60),(0.50,0.505),lw=1.7)
savefig(fig,"53_03_constraint_hierarchy.png")
plt.show()

## 17 Region First

Search over admissible sets, then choose the operating point.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5))
for ax in axes:
    clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
axes[0].set_title("Point-first search")
axes[1].set_title("Region-first search")
left=[("candidate",""),("objective",""),("gradient",""),("repeat","")]
right=[("candidate",""),("simulation",""),("Ω","admissible region"),("Ωc","largest component"),("distance","margin"),("x*","operation")]
vertical_chain(axes[0],0.5,0.72,0.14,left,box_w=0.55,box_h=0.08,title_size=10)
vertical_chain(axes[1],0.5,0.77,0.105,right,box_w=0.55,box_h=0.07,title_size=10)
axes[0].text(0.5,0.10,"optimizes a local score",ha="center",fontsize=11)
axes[1].text(0.5,0.10,"specifies topology before selecting a point",ha="center",fontsize=11)
fig.suptitle("Region First: Search Over Admissible Sets, Not Only Points",fontsize=18)
savefig(fig,"53_04_region_first_search.png")
plt.show()

## 23 Design Principles

The five principles are the compact statement of the notebook.

In [ ]:
fig, ax = plt.subplots(figsize=(12,6.5))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Five Design Principles")
principle_text=[
    ("1","Physics specifies admissibility."),
    ("2","Admissibility specifies topology."),
    ("3","Topology specifies robustness."),
    ("4","Distance specifies operation."),
    ("5","Operation specifies computation."),
]
for i,(n,t) in enumerate(principle_text):
    y=0.75-i*0.13
    ax.add_patch(Circle((0.18,y+0.04),0.04,fill=False,linewidth=2.2))
    ax.text(0.18,y+0.04,n,ha="center",va="center",fontsize=14,fontweight="bold")
    box(ax,0.27,y,0.56,0.08,t,"",lw=1.8,rounded=True,title_size=12)
ax.text(0.5,0.08,"Maximize the admissible connected region before selecting the operating point.",ha="center",fontsize=13)
savefig(fig,"53_05_five_design_principles.png")
plt.show()

## 29 Engineering Interpretation

The symbols are engineering objects, not only mathematical notation.

In [ ]:
interp=pd.DataFrame([
    ["Ω","admissible behaviors","what survives constraints"],
    ["Ωc","largest connected design family","what remains navigable"],
    ["d(∂Ωc)","robustness margin","distance to failure"],
    ["x*","operating point","chosen from robust interior"],
    ["J(D)","architecture comparison","summary of region + penalties"],
], columns=["Quantity","Engineering meaning","Use"])
interp.to_csv(DATA/"53_engineering_interpretation.csv",index=False)

fig, ax = plt.subplots(figsize=(11,3.9))
clean_axes(ax)
ax.set_title("Engineering Interpretation")
tbl=ax.table(cellText=interp.values,colLabels=interp.columns,cellLoc="center",loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1,1.6)
for (r,c),cell in tbl.get_celld().items():
    cell.set_linewidth(1)
    if r==0: cell.set_text_props(fontweight="bold")
savefig(fig,"53_06_engineering_interpretation_table.png")
plt.show()
interp

## 31 Architecture Ranking

Architectures are ranked by their path through specifications, not by a single isolated score.

In [ ]:
fig, ax = plt.subplots(figsize=(12,5.2))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Architecture Ranking as a Path Through Specifications")
architectures=[
    ("single cavity",0.18,["resources","Ω","partial Ωc","low margin","fragile x*"]),
    ("coupled resonators",0.34,["resources","Ω","connected Ωc","medium margin","usable x*"]),
    ("programmable mesh",0.50,["resources","larger Ω","connected Ωc","good margin","complex x*"]),
    ("hybrid chip",0.66,["resources","largest Ω","largest Ωc","best margin","robust x*"]),
]
for name,y,path in architectures:
    ax.text(0.05,y,name,ha="left",va="center",fontsize=10,fontweight="bold")
    xs=np.linspace(0.25,0.88,len(path))
    for i,p in enumerate(path):
        box(ax,xs[i]-0.055,y-0.04,0.11,0.08,p,"",rounded=True,lw=1.3,title_size=7)
        if i<len(path)-1:
            arrow(ax,(xs[i]+0.06,y),(xs[i+1]-0.06,y),lw=1.1)
ax.text(0.5,0.88,"Ranking follows the path that preserves the largest robust connected region.",ha="center",fontsize=12)
savefig(fig,"53_07_architecture_ranking_paths.png")
plt.show()

## 37 Constraint Lattice

The design object moves from physics to resources to constraints to admissible topology.

In [ ]:
fig, ax = plt.subplots(figsize=(8,7))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Constraint Lattice")
levels=[
    ("physics",0.86),
    ("resources",0.74),
    ("constraints",0.62),
    ("Ω",0.50),
    ("Ωc",0.38),
    ("distance",0.26),
    ("objective",0.14),
    ("x*",0.02),
]
for label,y in levels:
    box(ax,0.34,y,0.32,0.075,label,"",rounded=True,lw=2,title_size=12)
for (_,y1),(_,y2) in zip(levels[:-1],levels[1:]):
    arrow(ax,(0.50,y1),(0.50,y2+0.075),lw=1.8)
# lateral constraints
for txt,x,y in [("loss",0.13,0.58),("timing",0.13,0.45),("calibration",0.76,0.58),("control",0.76,0.45)]:
    box(ax,x,y,0.16,0.06,txt,"",rounded=True,lw=1.2,title_size=8)
    arrow(ax,(x+0.08,y),(0.50,0.50),lw=1.0,color="0.3")
savefig(fig,"53_08_constraint_lattice.png")
plt.show()

## 41 Why Topology Comes First

A good point near a failure boundary can be worse than a slightly lower score inside a robust connected region.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5))
for ax in axes:
    clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_aspect("equal")
axes[0].set_title("Region first")
axes[0].add_patch(Ellipse((0.5,0.5),0.65,0.38,fill=False,linewidth=2.2))
axes[0].add_patch(Ellipse((0.5,0.5),0.38,0.20,fill=False,linestyle="--",linewidth=1.8))
axes[0].scatter([0.5],[0.5],s=130)
axes[0].text(0.5,0.79,"large admissible family",ha="center",fontsize=11)
axes[0].text(0.5,0.36,"choose safest point",ha="center",fontsize=10)

axes[1].set_title("Point first")
axes[1].scatter([0.52],[0.52],s=130)
axes[1].add_patch(Circle((0.52,0.52),0.10,fill=False,linestyle="--",linewidth=1.8))
axes[1].add_patch(Rectangle((0.60,0.25),0.03,0.55,fill=True,alpha=0.2))
axes[1].text(0.52,0.72,"single optimum",ha="center",fontsize=11)
axes[1].text(0.66,0.55,"near boundary",ha="left",fontsize=10)
fig.suptitle("Why Topology Comes First",fontsize=18)
fig.text(0.5,0.05,"A point can score well while the surrounding admissible region is fragile.",ha="center",fontsize=12)
savefig(fig,"53_09_why_topology_comes_first.png")
plt.show()

## 47 Scientific Interpretation

The same region has physical, mathematical, and engineering meanings.

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(13,4.6))
titles=["Physics","Mathematics","Engineering"]
items=[
    [("allowed states","resources"),("loss budget","constraints"),("drive window","operation")],
    [("Ω","admissible set"),("Ωc","connected component"),("d(∂Ωc)","distance field")],
    [("design family","architecture"),("robust margin","stability"),("x*","execution point")],
]
for ax,title,rows in zip(axes,titles,items):
    clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title(title)
    for i,(t,s) in enumerate(rows):
        box(ax,0.14,0.68-i*0.22,0.72,0.12,t,s,rounded=True,lw=1.8,title_size=11,subtitle_size=8)
fig.suptitle("Scientific Interpretation",fontsize=18)
fig.text(0.5,0.05,"The same object appears as physics, mathematics, and engineering depending on the question asked.",ha="center",fontsize=12)
savefig(fig,"53_10_scientific_interpretation.png")
plt.show()

## 49 Implications

This section captures what changes when topology becomes the leading specification.

In [ ]:
fig, ax = plt.subplots(figsize=(11,5.2))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Implications")
implications=[
    "Topology is a specification.",
    "Objectives summarize topology.",
    "Distance transforms measure robustness.",
    "Optimization is downstream.",
    "Fault tolerance is selected from the robust region.",
]
for i,t in enumerate(implications):
    y=0.75-i*0.13
    ax.text(0.18,y,"✓",fontsize=18,ha="center",va="center")
    box(ax,0.25,y-0.04,0.58,0.08,t,"",rounded=True,lw=1.6,title_size=11)
savefig(fig,"53_11_implications.png")
plt.show()

## 53 Conclusion

The notebook ends with the reusable principle statement.

In [ ]:
fig, ax = plt.subplots(figsize=(10,5.8))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Conclusion")
ax.text(0.5,0.76,"Region-Constrained Design Search",ha="center",fontsize=24,fontweight="bold")
ax.text(0.5,0.58,"preserves admissible topology",ha="center",fontsize=18)
ax.text(0.5,0.48,"before",ha="center",fontsize=14)
ax.text(0.5,0.38,"selecting operating points",ha="center",fontsize=18)
for x,t in zip([0.18,0.38,0.60,0.82],["Specification","Robustness","Operation","Computation"]):
    box(ax,x-0.09,0.12,0.18,0.09,t,"",rounded=True,lw=1.8,title_size=10)
for x1,x2 in zip([0.27,0.47,0.69],[0.29,0.51,0.73]):
    arrow(ax,(x1,0.165),(x2,0.165),lw=1.6)
savefig(fig,"53_12_conclusion_manifesto.png")
plt.show()

## 59 The Region Is the Specification

The admissible region is the object to preserve.

In [ ]:
# Visualize the region as the specification.
fig, ax = plt.subplots(figsize=(8.5,5.7))
im=ax.imshow(Z,origin="lower",extent=[0,1,0,1],aspect="auto",vmin=0,vmax=1)
ax.contour(X,Y,Z,levels=[threshold],colors="black",linewidths=2.0)
ax.contour(X,Y,M["largest"].astype(float),levels=[0.5],colors="black",linewidths=2.8)
ax.scatter([M["x"]],[M["y"]],s=150,zorder=4)
ax.annotate("operating point\nchosen after topology",xy=(M["x"],M["y"]),xytext=(0.68,0.22),
            arrowprops=dict(arrowstyle="->",linewidth=2),fontsize=10)
ax.text(0.46,0.50,"admissible\nregion Ω",ha="center",fontsize=14)
ax.text(0.66,0.12,"largest connected\ncomponent Ωc",ha="center",fontsize=11)
ax.set_xlabel("drive"); ax.set_ylabel("loss")
ax.set_title("The Region Is the Specification")
fig.colorbar(im,ax=ax,label="admissibility score")
savefig(fig,"53_13_region_is_the_specification.png")
plt.show()

## 61 Distance Specifies Operation

The distance transform selects an operating point after topology has been preserved.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5.2))
dist_masked=np.where(M["largest"],M["dist"],np.nan)
im=ax.imshow(dist_masked,origin="lower",extent=[0,1,0,1],aspect="auto",vmin=0,vmax=1)
ax.contour(X,Y,Z,levels=[threshold],colors="black",linewidths=2.0)
ax.scatter([M["x"]],[M["y"]],s=160,zorder=5)
ax.annotate("maximum-margin\noperating point",xy=(M["x"],M["y"]),xytext=(0.69,0.19),
            arrowprops=dict(arrowstyle="->",linewidth=2),fontsize=10)
ax.set_xlabel("drive"); ax.set_ylabel("loss")
ax.set_title("Distance Specifies Operation")
fig.colorbar(im,ax=ax,label="distance to failure boundary")
savefig(fig,"53_14_distance_specifies_operation.png")
plt.show()

## 67 Export summary metadata

In [ ]:
summary = {
    "notebook": "53_design_principles_region_constrained_search",
    "thesis": "Physics specifies admissibility; admissibility specifies topology; topology specifies robustness; robustness specifies operation; operation specifies computation.",
    "principles": principles.to_dict(orient="records"),
    "pipeline": ["Physics", "Admissibility", "Topology Ωc", "Robustness d(∂Ωc)", "Operation x*", "Computation"],
    "figure_count": len(list(FIG.glob("*.png"))),
    "data_files": sorted([p.name for p in DATA.glob("*")]),
}
Path(DATA / "53_design_principles_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

## 71 Package and download outputs

In [ ]:
# ============================================================
# Package and download Notebook 53 outputs
# ============================================================
#
# Run this final cell after executing the notebook.
#
# It creates:
#   53_design_principles_outputs.zip
#
# containing:
#   outputs/notebook_53_design_principles/figures/*.png
#   outputs/notebook_53_design_principles/data/*.csv
#   outputs/notebook_53_design_principles/data/*.json
#   README_53_outputs.md
#
# In Google Colab, it triggers an automatic browser download.
# In Jupyter, it displays a clickable download link.

import shutil
from pathlib import Path
from IPython.display import FileLink, display

package_root = Path("outputs/notebook_53_design_principles")
figure_dir = package_root / "figures"
data_dir = package_root / "data"

package_root.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

figure_files = sorted(figure_dir.glob("*.png"))
data_files = sorted([p for p in data_dir.glob("*") if p.is_file()])

manifest = package_root / "README_53_outputs.md"
manifest.write_text(
    "# Notebook 53 Outputs\n\n"
    "Generated from `53_design_principles_region_constrained_search.ipynb`.\n\n"
    "## Figures\n"
    + ("\n".join(f"- figures/{p.name}" for p in figure_files) if figure_files else "- No figures found. Run all cells before packaging.")
    + "\n\n## Data\n"
    + ("\n".join(f"- data/{p.name}" for p in data_files) if data_files else "- No data files found. Run all cells before packaging.")
    + "\n",
    encoding="utf-8",
)

archive_path = Path(shutil.make_archive("53_design_principles_outputs", "zip", root_dir=package_root)).resolve()

print(f"Packaged outputs: {archive_path}")
print(f"Figures: {len(figure_files)}")
print(f"Data files: {len(data_files)}")

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    display(FileLink(str(archive_path)))